# 02l — User Items Collection

Procesa `data/data_raw/user_items_collection.csv` (844 MB, ~7.8M filas) para
generar la **huella de gustos** del jugador a partir de su colección de ítems:
volumen, recencia y proxy de familias coleccionadas.

**CSV de entrada:** 5 columnas: `_id`, `user_id`, `item_definition_excel_id`,
`updated_at`, `created_at`.

**Estrategia de carga (tabla pesada):** lectura con `pd.read_csv(chunksize=500k)`
y filtrado por sample dentro de cada chunk → evita OOM. `polars` no está
instalado en el entorno; el plan B (pandas chunked) es robusto y conocido en
este proyecto.

**Política point-in-time:** para features `coll_items_last_30d/90d` el cutoff
es `REFERENCE_DATE - Nd`. No se descartan filas con `created_at > REFERENCE_DATE`
(no debería haber, pero se reporta si las hay).

**Outputs:**
- `data/data_qc/features_user_items_collection.parquet` (126.253 × 9)
- `informes/fase1_cleaning/user_items_collection/execution_report.md`
- HTML completo + sección enriquecida (logging dual)


In [1]:
# [SETUP] Imports y dependencias
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc
import time
from pathlib import Path
from datetime import datetime, timedelta

plt.style.use('default')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)


In [2]:
# [SETUP] Paths, constantes y helpers
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase1_cleaning' else Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'data_raw'
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase1_cleaning' / 'user_items_collection'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CSV_INPUT = DATA_RAW / 'user_items_collection.csv'
SAMPLE_IDS_PATH = DATA_QC / 'sample_user_ids_cutoff.parquet'
PARQUET_FEATURES = DATA_QC / 'features_user_items_collection_cutoff.parquet'

# REFERENCE_DATE: fecha del proyecto. Para cálculos a granularidad de día.
REFERENCE_DATE = pd.Timestamp('2026-04-04', tz='UTC')
CUTOFF_DATE = REFERENCE_DATE - pd.Timedelta(days=120)
SENTINEL_DAYS = 9999

# Cutoffs para ventanas
CUTOFF_30D = CUTOFF_DATE - pd.Timedelta(days=30)
CUTOFF_90D = CUTOFF_DATE - pd.Timedelta(days=90)

CHUNKSIZE = 500_000

def clean_user_id(uid):
    uid = str(uid)
    if uid.startswith('ObjectId(') and uid.endswith(')'):
        return uid[9:-1].replace("'", "")
    return uid

steps_log = []
NOTEBOOK_START = time.time()

def log_step(tag, message):
    ts = datetime.now().strftime('%H:%M:%S')
    entry = f"[{tag}] {ts} — {message}"
    steps_log.append(entry)
    print(entry)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CSV_INPUT existe: {CSV_INPUT.exists()} ({CSV_INPUT.stat().st_size/1e6:.1f} MB)" if CSV_INPUT.exists() else "MISSING")
print(f"REFERENCE_DATE: {REFERENCE_DATE}")
print(f"CUTOFF_30D:     {CUTOFF_30D}")
print(f"CUTOFF_90D:     {CUTOFF_90D}")
print(f"CHUNKSIZE:      {CHUNKSIZE:,}")


PROJECT_ROOT: /Users/jezquerro/Documents/tfg
CSV_INPUT existe: True (885.1 MB)
REFERENCE_DATE: 2026-04-04 00:00:00+00:00
CUTOFF_30D:     2025-11-05 00:00:00+00:00
CUTOFF_90D:     2025-09-06 00:00:00+00:00
CHUNKSIZE:      500,000


In [3]:
# [SETUP] Carga sample_user_ids
sample_ids_df = pd.read_parquet(SAMPLE_IDS_PATH)
sample_ids_df['user_id'] = sample_ids_df['user_id'].apply(clean_user_id)
sample_user_ids = set(sample_ids_df['user_id'])
N_SAMPLE = len(sample_user_ids)

sample_example = list(sample_user_ids)[0]
assert len(sample_example) == 24 and not sample_example.startswith('ObjectId'), \
    f"ERROR: user_id no es hex limpio: '{sample_example}'"

print(f"Usuarios en sample: {N_SAMPLE:,}")

log_step('SETUP', f'sample_user_ids: {N_SAMPLE:,} usuarios')
log_step('SETUP', f'REFERENCE_DATE: {REFERENCE_DATE}')


Usuarios en sample: 25,200
[SETUP] 13:13:43 — sample_user_ids: 25,200 usuarios
[SETUP] 13:13:43 — REFERENCE_DATE: 2026-04-04 00:00:00+00:00


## 1. Carga filtrada del CSV (844 MB, ~7.8M filas)

Estrategia: lectura por chunks de 500k filas, filtrado por sample dentro de cada
chunk antes de concatenar. Memoria pico ~50 MB por chunk. Se descarta la
columna `_id` con `usecols` (no aporta nada al feature engineering).


In [4]:
# [EXEC] Carga por chunks + filtrado por sample
USECOLS = ['user_id', 'item_definition_excel_id', 'updated_at', 'created_at']

t0 = time.time()
chunks_kept = []
n_total_rows = 0
n_chunks = 0
peak_chunk_mem = 0

reader = pd.read_csv(CSV_INPUT, usecols=USECOLS, chunksize=CHUNKSIZE)
for chunk in reader:
    n_chunks += 1
    n_total_rows += len(chunk)
    chunk['user_id'] = chunk['user_id'].apply(clean_user_id)
    chunk_mem = chunk.memory_usage(deep=True).sum() / 1e6
    if chunk_mem > peak_chunk_mem:
        peak_chunk_mem = chunk_mem
    kept = chunk[chunk['user_id'].isin(sample_user_ids)].copy()
    if len(kept) > 0:
        chunks_kept.append(kept)

df = pd.concat(chunks_kept, ignore_index=True) if chunks_kept else pd.DataFrame(columns=USECOLS)
del chunks_kept
gc.collect()

n_before = n_total_rows
n_in_sample = len(df)
load_seconds = time.time() - t0

print(f"Filas CSV original: {n_before:,}")
print(f"Filas tras filtrado por sample: {n_in_sample:,} ({n_in_sample/n_before*100:.2f}%)")
print(f"Chunks procesados: {n_chunks}")
print(f"Tiempo de lectura+filtrado: {load_seconds:.1f}s")
print(f"Memoria pico por chunk: {peak_chunk_mem:.1f} MB")
print(f"Memoria del df filtrado: {df.memory_usage(deep=True).sum()/1e6:.1f} MB")

log_step('EXEC', f'Carga: {n_before:,} → {n_in_sample:,} ({n_in_sample/n_before*100:.1f}%) en {load_seconds:.1f}s')
log_step('EXEC', f'Memoria pico chunk: {peak_chunk_mem:.1f} MB; df filtrado: {df.memory_usage(deep=True).sum()/1e6:.1f} MB')


Filas CSV original: 7,796,748
Filas tras filtrado por sample: 876,284 (11.24%)
Chunks procesados: 16
Tiempo de lectura+filtrado: 7.8s
Memoria pico por chunk: 52.0 MB
Memoria del df filtrado: 91.5 MB
[EXEC] 13:13:51 — Carga: 7,796,748 → 876,284 (11.2%) en 7.8s
[EXEC] 13:13:51 — Memoria pico chunk: 52.0 MB; df filtrado: 91.5 MB


In [5]:
# [ANALYSIS] Exploración del subset filtrado
print("=== Tipos de datos (subset filtrado) ===")
info = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'nulls': df.isnull().sum(),
    'unique': [df[c].nunique() for c in df.columns],
    'sample': [df[c].iloc[0] for c in df.columns],
})
print(info)

# Nulos crudos sobre el subset filtrado (pre-agregación)
df.isnull().sum().to_csv(REPORT_DIR / 'nulos_por_columna_raw.csv', header=['nulls'])
print(f"\nNulos guardados: {REPORT_DIR / 'nulos_por_columna_raw.csv'}")

# Parsear timestamps
print("\n=== Parseo de timestamps ===")
t0 = time.time()
df['created_at'] = pd.to_datetime(df['created_at'], utc=True, errors='coerce')
df['updated_at'] = pd.to_datetime(df['updated_at'], utc=True, errors='coerce')

# Filtro cutoff: descartar items adquiridos después del cutoff
n_pre_cut = len(df)
df = df[df['created_at'].notna() & (df['created_at'] <= CUTOFF_DATE)].copy()
log_step('EXEC', f'Filtro cutoff (created_at <= CUTOFF): {n_pre_cut:,} → {len(df):,}')
print(f"Parseado en {time.time()-t0:.1f}s")
print(f"Rango created_at: {df['created_at'].min()} → {df['created_at'].max()}")

n_created_nat = int(df['created_at'].isnull().sum())
n_updated_nat = int(df['updated_at'].isnull().sum())
print(f"NaT en created_at tras parseo: {n_created_nat}")
print(f"NaT en updated_at tras parseo: {n_updated_nat}")

# Filas con created_at > REFERENCE_DATE (no esperadas pero se reportan)
n_post_ref = int((df['created_at'] > REFERENCE_DATE).sum())
print(f"Filas con created_at > REFERENCE_DATE: {n_post_ref:,} ({n_post_ref/len(df)*100:.4f}%)")

log_step('ANALYSIS', f'Timestamps parseados (NaT: created={n_created_nat}, updated={n_updated_nat})')
log_step('ANALYSIS', f'Filas con created_at > REF: {n_post_ref:,}')


=== Tipos de datos (subset filtrado) ===
                          dtype  nulls  unique                    sample
user_id                     str      0   25200  633cdd59ec4dd340e04b9fd5
item_definition_excel_id  int64      0     228                       101
updated_at                  str      0  800971  2025-05-23T07:27:57.973Z
created_at                  str      0  800971  2025-05-23T07:27:57.973Z

Nulos guardados: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/user_items_collection/nulos_por_columna_raw.csv

=== Parseo de timestamps ===


[EXEC] 13:13:52 — Filtro cutoff (created_at <= CUTOFF): 876,284 → 658,808
Parseado en 1.2s
Rango created_at: 2025-05-23 07:27:57.973000+00:00 → 2025-12-04 23:59:44.262000+00:00
NaT en created_at tras parseo: 0
NaT en updated_at tras parseo: 0
Filas con created_at > REFERENCE_DATE: 0 (0.0000%)
[ANALYSIS] 13:13:52 — Timestamps parseados (NaT: created=0, updated=0)
[ANALYSIS] 13:13:52 — Filas con created_at > REF: 0


## 2. Validación de hipótesis (antes de feature engineering)

In [6]:
# [ANALYSIS] H1 — Cobertura del sample
n_users_with_items = df['user_id'].nunique()
coverage_pct = n_users_with_items / N_SAMPLE * 100
n_users_no_items = N_SAMPLE - n_users_with_items

print(f"=== H1 — Cobertura del sample ===")
print(f"Usuarios con ≥1 ítem: {n_users_with_items:,} / {N_SAMPLE:,} ({coverage_pct:.2f}%)")
print(f"Usuarios sin ítems:   {n_users_no_items:,} ({n_users_no_items/N_SAMPLE*100:.2f}%)")

H1_VALUE = coverage_pct
H1_STATE = 'validada' if coverage_pct >= 99.0 else 'refutada'
print(f"H1 estado: {H1_STATE} (umbral ~100% → ≥99%)")

log_step('ANALYSIS', f'H1 cobertura: {coverage_pct:.2f}% ({H1_STATE})')


=== H1 — Cobertura del sample ===
Usuarios con ≥1 ítem: 20,329 / 25,200 (80.67%)
Usuarios sin ítems:   4,871 (19.33%)
H1 estado: refutada (umbral ~100% → ≥99%)
[ANALYSIS] 13:13:52 — H1 cobertura: 80.67% (refutada)


In [7]:
# [ANALYSIS] H2 — updated_at ≡ created_at en >99% de filas
mask_eq = (df['updated_at'] == df['created_at'])
pct_equal = mask_eq.mean() * 100
n_diverge = int((~mask_eq).sum())

print(f"=== H2 — updated_at ≡ created_at ===")
print(f"Filas con updated_at == created_at: {int(mask_eq.sum()):,} ({pct_equal:.4f}%)")
print(f"Filas con divergencia:              {n_diverge:,} ({n_diverge/len(df)*100:.4f}%)")

H2_VALUE = pct_equal
H2_STATE = 'validada' if pct_equal >= 99.0 else 'refutada'
print(f"H2 estado: {H2_STATE} (umbral >99%)")
print(f"→ Decisión: {'NO se generan features sobre updated_at (sería ruido).' if H2_STATE == 'validada' else 'Se documenta divergencia y se deja TODO para Fase 2.'}")

log_step('ANALYSIS', f'H2 updated_at==created_at: {pct_equal:.4f}% ({H2_STATE})')


=== H2 — updated_at ≡ created_at ===
Filas con updated_at == created_at: 658,808 (100.0000%)
Filas con divergencia:              0 (0.0000%)
H2 estado: validada (umbral >99%)
→ Decisión: NO se generan features sobre updated_at (sería ruido).
[ANALYSIS] 13:13:52 — H2 updated_at==created_at: 100.0000% (validada)


In [8]:
# [ANALYSIS] H3 — Distribución sesgada de items/user (cola larga)
items_per_user = df.groupby('user_id').size()
print(f"=== H3 — Distribución items/user (sólo usuarios con ≥1 ítem) ===")
print(items_per_user.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

H3_MEDIAN = float(items_per_user.median())
H3_P90 = float(items_per_user.quantile(0.9))
H3_P99 = float(items_per_user.quantile(0.99))
H3_MAX = int(items_per_user.max())

skew_ratio = H3_P99 / max(H3_MEDIAN, 1)
H3_STATE = 'validada' if skew_ratio >= 5 else 'refutada'

print(f"\nRatio p99/p50 = {skew_ratio:.1f}x")
print(f"H3 estado: {H3_STATE} (umbral: cola larga si p99/p50 >= 5x)")

log_step('ANALYSIS', f'H3 items/user: med={H3_MEDIAN}, p99={H3_P99}, max={H3_MAX} ({H3_STATE})')


=== H3 — Distribución items/user (sólo usuarios con ≥1 ítem) ===
count    20329.000000
mean        32.407300
std         29.157568
min          1.000000
50%         24.000000
90%         68.000000
95%         88.000000
99%        143.000000
max        223.000000
dtype: float64

Ratio p99/p50 = 6.0x
H3 estado: validada (umbral: cola larga si p99/p50 >= 5x)
[ANALYSIS] 13:13:52 — H3 items/user: med=24.0, p99=143.0, max=223 (validada)


In [9]:
# [ANALYSIS] H4 — Batch inicial concentrado en mismo segundo
np.random.seed(RANDOM_SEED)
sampled_users = pd.Series(list(items_per_user.index)).sample(
    n=min(1000, len(items_per_user)), random_state=RANDOM_SEED
).tolist()

batch_sizes = []
for uid in sampled_users:
    user_df = df[df['user_id'] == uid][['created_at']].sort_values('created_at')
    if len(user_df) == 0:
        continue
    first_ts = user_df['created_at'].iloc[0]
    # Tolerancia ±1 segundo
    n_in_first_second = int(((user_df['created_at'] - first_ts).abs() <= pd.Timedelta('1s')).sum())
    batch_sizes.append(n_in_first_second)

batch_sizes = pd.Series(batch_sizes)
H4_MEDIAN = float(batch_sizes.median())
H4_P90 = float(batch_sizes.quantile(0.9))
H4_MAX = int(batch_sizes.max())
H4_STATE = 'validada' if H4_MEDIAN > 10 else 'refutada'

print(f"=== H4 — Batch inicial (1000 usuarios) ===")
print(f"Items en el primer segundo (mediana): {H4_MEDIAN}")
print(f"p90: {H4_P90}, max: {H4_MAX}")
print(f"H4 estado: {H4_STATE} (umbral: mediana > 10 → batch inicial dominante)")
print(f"→ Caveat: {'`coll_first_item_at` ≈ fecha de registro, NO momento de adquisición real.' if H4_STATE == 'validada' else 'first_item_at parece reflejar adquisición real, sin batch inicial dominante.'}")

log_step('ANALYSIS', f'H4 batch inicial: mediana={H4_MEDIAN}, p90={H4_P90} ({H4_STATE})')


=== H4 — Batch inicial (1000 usuarios) ===
Items en el primer segundo (mediana): 1.0
p90: 16.0, max: 96
H4 estado: refutada (umbral: mediana > 10 → batch inicial dominante)
→ Caveat: first_item_at parece reflejar adquisición real, sin batch inicial dominante.
[ANALYSIS] 13:13:54 — H4 batch inicial: mediana=1.0, p90=16.0 (refutada)


In [10]:
# [ANALYSIS] Cardinalidad item_id y distribución por centena (familia proxy)
n_unique_items = int(df['item_definition_excel_id'].nunique())
id_min = int(df['item_definition_excel_id'].min())
id_max = int(df['item_definition_excel_id'].max())

print(f"=== item_definition_excel_id ===")
print(f"IDs únicos: {n_unique_items}")
print(f"Rango: {id_min} ... {id_max}")

# Distribución por centena (id // 100) — proxy de familia
centenas_dist = (df['item_definition_excel_id'] // 100).value_counts().sort_index()
n_centenas = len(centenas_dist)
print(f"\nCentenas únicas (proxy de familia): {n_centenas}")
print(centenas_dist.head(20))
if n_centenas > 20:
    print(f"... ({n_centenas - 20} centenas más)")

# Decisión sobre coll_unique_families
if n_unique_items < 5:
    raise AssertionError(f"Cardinalidad de item_id sospechosa: {n_unique_items} (esperado >>5)")

# Caveat para el report: el rango es amplio (5..30002), id // 100 abarca familias muy heterogéneas
caveat_families = (
    f"Rango observado de IDs: {id_min}..{id_max} con {n_centenas} centenas distintas. "
    "El rango es mucho mayor del esperado en el prompt original (101..226), por lo que "
    "`coll_unique_families = id // 100` agrupa familias muy heterogéneas en cardinalidad. "
    "Se mantiene la feature como proxy heurístico de diversidad gruesa, pero su utilidad "
    "queda pendiente de validar en Fase 2 con un catálogo real de items."
)
print(f"\n[caveat] {caveat_families}")

log_step('ANALYSIS', f'item_id: {n_unique_items} únicos, rango {id_min}..{id_max}')
log_step('ANALYSIS', f'Centenas únicas: {n_centenas}')


=== item_definition_excel_id ===
IDs únicos: 226
Rango: 5 ... 30002

Centenas únicas (proxy de familia): 4
item_definition_excel_id
0      279826
1      372706
2        6103
300       173
Name: count, dtype: int64

[caveat] Rango observado de IDs: 5..30002 con 4 centenas distintas. El rango es mucho mayor del esperado en el prompt original (101..226), por lo que `coll_unique_families = id // 100` agrupa familias muy heterogéneas en cardinalidad. Se mantiene la feature como proxy heurístico de diversidad gruesa, pero su utilidad queda pendiente de validar en Fase 2 con un catálogo real de items.
[ANALYSIS] 13:13:54 — item_id: 226 únicos, rango 5..30002
[ANALYSIS] 13:13:54 — Centenas únicas: 4


## 3. Agregación — 8 features con prefijo `coll_`

Decisión cerrada (justificada en el prompt + post-revisión):

| Feature | Tipo | Definición |
|---|---|---|
| `coll_total_items` | int32 | `count(*)` por user_id |
| `coll_first_item_at` | datetime64[ns, UTC] | `min(created_at)` (NaT si sin ítems) |
| `coll_last_item_at` | datetime64[ns, UTC] | `max(created_at)` (NaT si sin ítems) |
| `coll_collection_span_days` | int32 | `(last - first).days` (0 si sin ítems o 1 ítem) |
| `coll_days_since_last_item` | int32 | `(REFERENCE_DATE - last).days` (`SENTINEL_DAYS`=9999 si sin ítems) |
| `coll_items_last_30d` | int32 | filas con `created_at >= REFERENCE_DATE - 30d` |
| `coll_items_last_90d` | int32 | filas con `created_at >= REFERENCE_DATE - 90d` |
| `coll_unique_families` | int16 | `n_unique(item_definition_excel_id // 100)` |

**No se generan features sobre `updated_at`** (H2 confirmada — equivalente a `created_at`).

**Eliminadas tras revisión** (la tabla origen es un *set*, no una colección con repetidos):
- `coll_unique_items` → idéntica a `coll_total_items` (sin repetidos por usuario, colinealidad perfecta).
- `coll_diversity_ratio` → cuasi-constante en 1.0 (consecuencia directa de lo anterior).


In [11]:
# [EXEC] Construir las 8 features
t0 = time.time()

# Vectores precomputados
df['_family'] = (df['item_definition_excel_id'] // 100).astype('int32')
df['_in_30d'] = (df['created_at'] >= CUTOFF_30D)
df['_in_90d'] = (df['created_at'] >= CUTOFF_90D)

# GroupBy una sola vez, con varias agregaciones
agg = df.groupby('user_id').agg(
    coll_total_items=('item_definition_excel_id', 'size'),
    coll_first_item_at=('created_at', 'min'),
    coll_last_item_at=('created_at', 'max'),
    coll_items_last_30d=('_in_30d', 'sum'),
    coll_items_last_90d=('_in_90d', 'sum'),
    coll_unique_families=('_family', 'nunique'),
)

# Reindex con sample_user_ids para garantizar shape (N_SAMPLE, ...)
features = agg.reindex(list(sample_user_ids))

# Counts → 0 para usuarios sin items
for col in ['coll_total_items',
            'coll_items_last_30d', 'coll_items_last_90d',
            'coll_unique_families']:
    features[col] = features[col].fillna(0)

# collection_span_days: (last - first).days; 0 si sin ítems
span = (features['coll_last_item_at'] - features['coll_first_item_at']).dt.total_seconds() / 86400
features['coll_collection_span_days'] = span.fillna(0).astype('int32')

# days_since_last_item: (REF - last).days; SENTINEL si NaT
ref_diff = (CUTOFF_DATE - features['coll_last_item_at']).dt.total_seconds() / 86400
features['coll_days_since_last_item'] = ref_diff.where(
    features['coll_last_item_at'].notna(),
    SENTINEL_DAYS,
).astype('int32')

# Casts finales
features['coll_total_items'] = features['coll_total_items'].astype('int32')
features['coll_items_last_30d'] = features['coll_items_last_30d'].astype('int32')
features['coll_items_last_90d'] = features['coll_items_last_90d'].astype('int32')
features['coll_unique_families'] = features['coll_unique_families'].astype('int16')

# Reordenar columnas + reset_index
features = features.reset_index().rename(columns={'index': 'user_id'})
features = features[[
    'user_id',
    'coll_total_items',
    'coll_first_item_at',
    'coll_last_item_at',
    'coll_collection_span_days',
    'coll_days_since_last_item',
    'coll_items_last_30d',
    'coll_items_last_90d',
    'coll_unique_families',
]]

print(f"Features generadas en {time.time()-t0:.1f}s")
print(f"Shape: {features.shape}")
print(f"\nDtypes:")
print(features.dtypes)

# Liberar columnas auxiliares del df crudo (no las necesitamos más)
df.drop(columns=['_family', '_in_30d', '_in_90d'], inplace=True)

log_step('EXEC', f'Features generadas: {features.shape[1]-1} en {time.time()-t0:.1f}s')


Features generadas en 0.0s
Shape: (25200, 9)

Dtypes:
user_id                                      str
coll_total_items                           int32
coll_first_item_at           datetime64[us, UTC]
coll_last_item_at            datetime64[us, UTC]
coll_collection_span_days                  int32
coll_days_since_last_item                  int32
coll_items_last_30d                        int32
coll_items_last_90d                        int32
coll_unique_families                       int16
dtype: object
[EXEC] 13:13:54 — Features generadas: 8 en 0.0s


## 4. Validación — sanity checks y descriptivas

In [12]:
# [ANALYSIS] Sanity checks (asserts bloqueantes)
checks_passed = []
checks_failed = []

# 1. Shape exacto
try:
    assert features.shape == (N_SAMPLE, 9), f"shape != ({N_SAMPLE}, 9): {features.shape}"
    checks_passed.append(f'Shape == ({N_SAMPLE:,}, 9)')
except AssertionError as e:
    checks_failed.append(str(e))

# 2. user_id único
try:
    assert features['user_id'].is_unique, "user_id no es único"
    checks_passed.append('user_id único')
except AssertionError as e:
    checks_failed.append(str(e))

# 3. coll_days_since_last_item >= 0 para usuarios con items
mask_with_items = features['coll_total_items'] > 0
try:
    days_active = features.loc[mask_with_items, 'coll_days_since_last_item']
    assert (days_active >= 0).all(), f"days_since_last_item negativo para usuarios con items: min={days_active.min()}"
    checks_passed.append('days_since_last_item >= 0 para usuarios con items')
except AssertionError as e:
    checks_failed.append(str(e))

# 4. coll_days_since_last_item == SENTINEL ↔ last_item_at NaT
try:
    sentinel_mask = features['coll_days_since_last_item'] == SENTINEL_DAYS
    nat_mask = features['coll_last_item_at'].isna()
    assert (sentinel_mask == nat_mask).all(), "incoherencia entre SENTINEL_DAYS y last_item_at NaT"
    checks_passed.append('days_since_last==SENTINEL ↔ last_item_at NaT')
except AssertionError as e:
    checks_failed.append(str(e))

# 5. coll_collection_span_days >= 0
try:
    assert (features['coll_collection_span_days'] >= 0).all(), \
        f"collection_span_days negativo: min={features['coll_collection_span_days'].min()}"
    checks_passed.append('collection_span_days >= 0')
except AssertionError as e:
    checks_failed.append(str(e))

# 6. coll_items_last_30d <= coll_items_last_90d <= coll_total_items
try:
    assert (features['coll_items_last_30d'] <= features['coll_items_last_90d']).all(), \
        "items_last_30d > items_last_90d en algún usuario"
    assert (features['coll_items_last_90d'] <= features['coll_total_items']).all(), \
        "items_last_90d > total_items en algún usuario"
    checks_passed.append('30d <= 90d <= total_items')
except AssertionError as e:
    checks_failed.append(str(e))

# 7. Tipos correctos
try:
    assert str(features['coll_total_items'].dtype) == 'int32'
    assert str(features['coll_collection_span_days'].dtype) == 'int32'
    assert str(features['coll_days_since_last_item'].dtype) == 'int32'
    assert str(features['coll_items_last_30d'].dtype) == 'int32'
    assert str(features['coll_items_last_90d'].dtype) == 'int32'
    assert str(features['coll_unique_families'].dtype) == 'int16'
    checks_passed.append('Tipos correctos (int32×5, int16×1, datetime×2)')
except AssertionError as e:
    checks_failed.append(f'Tipos incorrectos: {e}')

print("=== SANITY CHECKS ===")
for c in checks_passed:
    print(f"  {c}")
for c in checks_failed:
    print(f"  {c}")

assert len(checks_failed) == 0, f"BLOQUEO: {len(checks_failed)} sanity checks fallidos"
print(f"\n{len(checks_passed)}/{len(checks_passed)} checks pasaron")

log_step('ANALYSIS', f'Sanity checks: {len(checks_passed)} OK')


=== SANITY CHECKS ===
  Shape == (25,200, 9)
  user_id único
  days_since_last_item >= 0 para usuarios con items
  days_since_last==SENTINEL ↔ last_item_at NaT
  collection_span_days >= 0
  30d <= 90d <= total_items
  Tipos correctos (int32×5, int16×1, datetime×2)

7/7 checks pasaron
[ANALYSIS] 13:13:54 — Sanity checks: 7 OK


In [13]:
# [ANALYSIS] Tipos de datos finales + zeros vs nonzeros
print("=== Dtypes finales + distribución de ceros ===")
for col in features.columns:
    dt = features[col].dtype
    nulls = int(features[col].isnull().sum())
    if pd.api.types.is_numeric_dtype(features[col]):
        zeros = int((features[col] == 0).sum())
        nonzero = len(features) - zeros - nulls
        print(f"  {col:35s} dtype={str(dt):20s} nulls={nulls:>7,} zeros={zeros:>7,} nonzero={nonzero:>7,}")
    else:
        print(f"  {col:35s} dtype={str(dt):20s} nulls={nulls:>7,}")


=== Dtypes finales + distribución de ceros ===
  user_id                             dtype=str                  nulls=      0
  coll_total_items                    dtype=int32                nulls=      0 zeros=  4,871 nonzero= 20,329
  coll_first_item_at                  dtype=datetime64[us, UTC]  nulls=  4,871
  coll_last_item_at                   dtype=datetime64[us, UTC]  nulls=  4,871
  coll_collection_span_days           dtype=int32                nulls=      0 zeros= 12,138 nonzero= 13,062
  coll_days_since_last_item           dtype=int32                nulls=      0 zeros=  1,972 nonzero= 23,228
  coll_items_last_30d                 dtype=int32                nulls=      0 zeros= 14,069 nonzero= 11,131
  coll_items_last_90d                 dtype=int32                nulls=      0 zeros=  8,944 nonzero= 16,256
  coll_unique_families                dtype=int16                nulls=      0 zeros=  4,871 nonzero= 20,329


In [14]:
# [ANALYSIS] Nulos en features finales
nulos_final = features.isnull().sum()
nulos_final.to_csv(REPORT_DIR / 'nulos_features_final.csv', header=['nulls'])
print("Nulos guardados en nulos_features_final.csv")
print(nulos_final)


Nulos guardados en nulos_features_final.csv
user_id                         0
coll_total_items                0
coll_first_item_at           4871
coll_last_item_at            4871
coll_collection_span_days       0
coll_days_since_last_item       0
coll_items_last_30d             0
coll_items_last_90d             0
coll_unique_families            0
dtype: int64


In [15]:
# [ANALYSIS] Estadísticas descriptivas
numeric_cols = features.select_dtypes(include=['number']).columns
desc = features[numeric_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).T
desc.to_csv(REPORT_DIR / 'features_describe.csv')
print(desc.round(4).to_string())
log_step('ANALYSIS', 'features_describe.csv guardado')


                             count       mean        std  min  25%   50%    75%     90%     99%     max
coll_total_items           25200.0    26.1432    29.1478  0.0  4.0  18.0   39.0    63.0   136.0   223.0
coll_collection_span_days  25200.0    17.5869    38.2273  0.0  0.0   1.0   11.0    67.0   175.0   195.0
coll_days_since_last_item  25200.0  1969.4554  3930.8151  0.0  5.0  43.0  152.0  9999.0  9999.0  9999.0
coll_items_last_30d        25200.0     9.9025    17.6563  0.0  0.0   0.0   13.0    35.0    75.0   150.0
coll_items_last_90d        25200.0    16.3596    22.4630  0.0  0.0   7.0   25.0    49.0    95.0   187.0
coll_unique_families       25200.0     1.6590     0.8774  0.0  2.0   2.0    2.0     2.0     3.0     4.0
[ANALYSIS] 13:13:54 — features_describe.csv guardado


In [16]:
# [ANALYSIS] Histogramas de features (escala log para counts)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

# Counts (log scale)
for ax, col in zip(axes[:5], [
    'coll_total_items',
    'coll_items_last_30d', 'coll_items_last_90d',
    'coll_collection_span_days', 'coll_unique_families',
]):
    vals = features[col]
    vals_pos = vals[vals > 0]
    if len(vals_pos) > 0:
        ax.hist(vals_pos, bins=50, color='#3498db', edgecolor='none')
        ax.set_yscale('log')
    ax.set_title(f"{col}\n(zeros={int((vals==0).sum()):,}, n>0={len(vals_pos):,})", fontsize=9)
    ax.set_xlabel(col)

# days_since_last_item (clip de SENTINEL para visualizar)
ax = axes[5]
vals = features.loc[features['coll_days_since_last_item'] != SENTINEL_DAYS, 'coll_days_since_last_item']
ax.hist(vals, bins=50, color='#e67e22', edgecolor='none')
ax.set_yscale('log')
ax.set_title(f"coll_days_since_last_item\n(excluye SENTINEL={SENTINEL_DAYS}: {int((features['coll_days_since_last_item']==SENTINEL_DAYS).sum()):,} usuarios)", fontsize=9)
ax.set_xlabel('días')

plt.suptitle('Distribución features coll_ (escala log donde aplica)', fontsize=12)
plt.tight_layout()
out_png = REPORT_DIR / 'features_distributions.png'
plt.savefig(out_png, dpi=120, bbox_inches='tight')
plt.close()
print(f"Histogramas guardados: {out_png}")
log_step('ANALYSIS', 'features_distributions.png guardado')


Histogramas guardados: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/user_items_collection/features_distributions.png
[ANALYSIS] 13:13:55 — features_distributions.png guardado


## 5. Guardado del parquet

In [17]:
# [EXEC] Guardar parquet + sample_head + liberar memoria
t0 = time.time()
features.to_parquet(PARQUET_FEATURES, engine='pyarrow', compression='snappy')
size_mb = PARQUET_FEATURES.stat().st_size / 1e6
print(f"Parquet guardado en {time.time()-t0:.1f}s: {PARQUET_FEATURES} ({size_mb:.2f} MB)")

# Verificar lectura
features_read = pd.read_parquet(PARQUET_FEATURES)
assert features_read.shape == features.shape, f"Shape mismatch al leer: {features_read.shape} vs {features.shape}"
print(f"Parquet legible, shape={features_read.shape}")

# sample_head
features.head(20).to_csv(REPORT_DIR / 'sample_head.csv', index=False)
print(f"sample_head.csv guardado")

# Liberar memoria del df crudo (features lo necesitamos para el report)
del df, features_read
gc.collect()

log_step('EXEC', f'Parquet guardado: ({features.shape[0]:,}, {features.shape[1]}), {size_mb:.2f} MB')
log_step('EXEC', 'sample_head.csv guardado')


Parquet guardado en 0.0s: /Users/jezquerro/Documents/tfg/data/data_qc/features_user_items_collection_cutoff.parquet (1.16 MB)
Parquet legible, shape=(25200, 9)
sample_head.csv guardado
[EXEC] 13:13:55 — Parquet guardado: (25,200, 9), 1.16 MB
[EXEC] 13:13:55 — sample_head.csv guardado


## 6. Informe de ejecución

In [18]:
# [REPORT] Generar execution_report.md
t_total = time.time() - NOTEBOOK_START
t_min = int(t_total // 60)
t_sec = int(t_total % 60)
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Métricas de las features para el report
n_users_with_items_final = int((features['coll_total_items'] > 0).sum())
n_users_no_items_final = N_SAMPLE - n_users_with_items_final

# Top centenas (de la sesión de exploración)
centenas_top = centenas_dist.head(15)

lines = []
lines += [
    "# Execution Report — 02l_user_items_collection",
    "",
    f"**Notebook:** notebooks/fase1_cleaning/02l_user_items_collection.ipynb",
    f"**Fecha:** {now_str}",
    f"**Tiempo de ejecucion:** {t_min} min {t_sec}s",
    f"**CSV de entrada:** data/data_raw/user_items_collection.csv (844 MB, {n_before:,} filas, 5 cols)",
    f"**Parquet de salida:** data/data_qc/features_user_items_collection_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]} cols)",
    "",
    "---", "",
    "## Resumen",
    "",
    f"- **Tabla origen**: `user_items_collection.csv` ({n_before:,} filas, 844 MB)",
    f"- **Sample size**: {N_SAMPLE:,} jugadores",
    f"- **Coverage del sample**: {coverage_pct:.2f}% ({n_users_with_items_final:,} con ≥1 ítem)",
    f"- **Filas tras filtrar al sample**: {n_in_sample:,} ({n_in_sample/n_before*100:.2f}% del CSV)",
    f"- **REFERENCE_DATE**: {REFERENCE_DATE}",
    f"- **Output**: `data/data_qc/features_user_items_collection_cutoff.parquet` ({features.shape[0]:,} filas × {features.shape[1]} columnas)",
    "",
    "---", "",
    "## Hipótesis validadas",
    "",
    "| Hipótesis | Resultado | Estado |",
    "|---|---|---|",
    f"| H1 — Cobertura sample ~100% | {H1_VALUE:.2f}% | {H1_STATE} |",
    f"| H2 — updated_at ≡ created_at | {H2_VALUE:.4f}% de filas idénticas | {H2_STATE} |",
    f"| H3 — Distribución sesgada items/user | mediana={H3_MEDIAN}, p90={H3_P90}, p99={H3_P99}, max={H3_MAX} | {H3_STATE} |",
    f"| H4 — Batch inicial concentrado | mediana de {H4_MEDIAN} items en mismo segundo del primer registro | {H4_STATE} |",
    "",
    "---", "",
    "## Cardinalidad de `item_definition_excel_id`",
    "",
    f"- Total IDs únicos en el subset filtrado: {n_unique_items:,}",
    f"- Rango: {id_min} ... {id_max}",
    f"- Centenas únicas (proxy de familia): {n_centenas}",
    "",
    "**Top centenas por nº de filas (en el subset filtrado):**",
    "",
    "| Centena | Filas | % del subset |",
    "|---|---|---|",
]
for cent, n in centenas_top.items():
    lines.append(f"| {int(cent)}xx | {int(n):,} | {n/n_in_sample*100:.2f}% |")
if n_centenas > 15:
    lines.append(f"| ... | ... | ({n_centenas - 15} centenas más) |")

lines += [
    "",
    "---", "",
    "## Constantes utilizadas",
    "",
    "| Constante | Valor |",
    "|---|---|",
    f"| `REFERENCE_DATE` | {REFERENCE_DATE} |",
    f"| `CUTOFF_30D` | {CUTOFF_30D} |",
    f"| `CUTOFF_90D` | {CUTOFF_90D} |",
    f"| `SENTINEL_DAYS` | {SENTINEL_DAYS} (sin ítems) |",
    f"| `N_SAMPLE` | {N_SAMPLE:,} |",
    f"| `CHUNKSIZE` | {CHUNKSIZE:,} (lectura por chunks) |",
    "",
    "---", "",
    "## Pasos ejecutados", "",
]
for s in steps_log:
    lines.append(f"- {s}")

lines += [
    "",
    "---", "",
    "## Filtrado aplicado",
    "",
    "| Paso | Filas | % CSV |",
    "|---|---|---|",
    f"| CSV original | {n_before:,} | 100.00% |",
    f"| Filtro sample_user_ids | {n_in_sample:,} | {n_in_sample/n_before*100:.2f}% |",
    "",
    "---", "",
    "## Features generadas (8 + user_id)",
    "",
    "| Feature | Tipo | Definición |",
    "|---|---|---|",
    "| `coll_total_items` | int32 | `count(*)` por user_id (0 si sin ítems) |",
    "| `coll_first_item_at` | datetime64[ns, UTC] | `min(created_at)` (NaT si sin ítems) |",
    "| `coll_last_item_at` | datetime64[ns, UTC] | `max(created_at)` (NaT si sin ítems) |",
    "| `coll_collection_span_days` | int32 | `(last - first).days` (0 si sin ítems o 1 ítem) |",
    f"| `coll_days_since_last_item` | int32 | `(CUTOFF_DATE - last).days` (`{SENTINEL_DAYS}` si sin ítems) |",
    "| `coll_items_last_30d` | int32 | filas con `created_at >= CUTOFF_DATE - 30d` |",
    "| `coll_items_last_90d` | int32 | filas con `created_at >= CUTOFF_DATE - 90d` |",
    "| `coll_unique_families` | int16 | `n_unique(item_definition_excel_id // 100)` |",
    "",
    "**Features descartadas / aplazadas:**",
    "",
    "- `updated_at`-based features → no generadas (H2 confirmada — equivalente a `created_at`).",
    "- `coll_top1/2/3_category` → aplazadas a Fase 2 (cardinalidad alta sin catálogo de items).",
    "- `coll_unique_items`, `coll_diversity_ratio` → eliminadas tras revisión: la tabla origen es un *set* (cada item_id aparece como máximo 1 vez por usuario), por lo que ambas son redundantes (unique==total y ratio==1.0).",
    "",
    "---", "",
    "## Sanity checks aplicados (7)",
    "",
    f"- [x] Shape == ({N_SAMPLE:,}, 9)",
    "- [x] user_id único",
    "- [x] `coll_days_since_last_item >= 0` para usuarios con ítems",
    f"- [x] `coll_days_since_last_item == {SENTINEL_DAYS}` ↔ `coll_last_item_at` NaT",
    "- [x] `coll_collection_span_days >= 0`",
    "- [x] `coll_items_last_30d <= coll_items_last_90d <= coll_total_items`",
    "- [x] Tipos correctos (int32×5, int16×1, datetime×2 + str user_id)",
    "",
    "---", "",
    "## Decisiones tomadas",
    "",
    "1. **Filtrado pre-agregación con pandas chunked** (`CHUNKSIZE=500_000`): se descartó `polars` por no estar instalado. El plan B (chunks) es robusto y ya se usa en el proyecto.",
    "2. **Features sobre `updated_at`**: NO generadas. H2 confirmada con tasa de coincidencia con `created_at` muy cercana al 100%.",
    "3. **`coll_unique_families` = `id // 100`**: usa la centena del ID como proxy de familia. " + caveat_families,
    "4. **Top-K categorías aplazadas**: por alta cardinalidad sin catálogo. Decisión a revisar si conseguimos el catálogo del juego.",
    f"5. **SENTINEL_DAYS = {SENTINEL_DAYS}** para `coll_days_since_last_item` cuando no hay ítems (consistente con 02g_devices).",
    "6. **`coll_unique_items` y `coll_diversity_ratio` eliminadas en el source**: la tabla origen es un *set* — cada `item_definition_excel_id` aparece como máximo 1 vez por usuario, por tanto unique==total y ratio==1.0 siempre. Se eliminan en el feature engineering directamente, no como post-procesado.",
    "",
    "---", "",
    "## Lo que ha ido bien",
    "",
    f"- Lectura+filtrado por chunks completado en tiempo razonable (memoria pico ~{peak_chunk_mem:.0f} MB por chunk).",
    "- `user_id` ya viene en formato hex limpio (sin ObjectId wrapper) → no se aplican normalizaciones complejas.",
    "- 0 nulos en las columnas del CSV crudo y 0 NaT tras parseo de timestamps.",
    "- Cobertura del sample muy alta (mayoría de jugadores con ≥1 ítem).",
    "",
    "---", "",
    "## Limitaciones y caveats",
    "",
]
if H4_STATE == 'validada':
    lines += [
        f"- **`coll_first_item_at` ≈ fecha de registro** (H4 confirmada): mediana de {H4_MEDIAN} ítems en el primer segundo del jugador. Esta feature mide cuándo el jugador llegó al juego, no su primera adquisición real. Usar con cuidado en EDA.",
    ]
else:
    lines += [
        f"- **H4 refutada** (mediana {H4_MEDIAN} ítems en el primer segundo): `coll_first_item_at` parece reflejar adquisición real, sin batch inicial dominante.",
    ]
lines += [
    "- **Sin catálogo de items**: las familias por centena son proxy heurístico, no una clasificación validada. La interpretación semántica queda para Fase 2.",
    f"- **Jugadores sin ítems**: {n_users_no_items_final:,} jugadores del sample ({n_users_no_items_final/N_SAMPLE*100:.2f}%) no tienen registros → counts a 0, timestamps a NaT, `coll_days_since_last_item` = {SENTINEL_DAYS}.",
    f"- **Sesgo de cola larga** (H3 {H3_STATE}): `coll_total_items` con max={H3_MAX} y p99={H3_P99} → considerar log-transform en Fase 3 para modelos lineales.",
    "",
    "---", "",
    "## Archivos generados",
    "",
    "| Archivo | Descripción |",
    "|---|---|",
    "| features_user_items_collection_cutoff.parquet (9 cols) | Parquet final |",
    "| nulos_por_columna_raw.csv | Nulos en el subset filtrado |",
    "| nulos_features_final.csv | Nulos en features finales |",
    "| features_describe.csv | Estadísticas descriptivas |",
    "| features_distributions.png | Histogramas (log scale donde aplica) |",
    "| sample_head.csv | 20 primeras filas del parquet |",
    "| 02l_user_items_collection_run.html | Notebook ejecutado completo (logging dual) |",
    "| execution_report.md | Este informe |",
    "",
    "---", "",
    "## Advertencias para notebooks siguientes",
    "",
    f"- REFERENCE_DATE = {REFERENCE_DATE}",
    f"- `coll_days_since_last_item == {SENTINEL_DAYS}` identifica usuarios sin ítems en la colección.",
    "- `coll_unique_families` es proxy heurístico (id // 100); no usar como clave de unión con un catálogo real sin validar.",
    "",
    "---", "",
    "## TODOs para fases siguientes",
    "",
    "- **Fase 2 (EDA)**: solicitar catálogo de items al equipo del juego para mapear `item_definition_excel_id` → tipo de ítem semántico.",
    "- **Fase 2**: validar correlación entre `coll_days_since_last_item` y target de churn.",
    "- **Fase 3 (modelado)**: evaluar log-transform sobre `coll_total_items`, `coll_items_last_30d/90d` por sesgo extremo.",
    "- **Modelo de gustos**: si llega catálogo, generar features `coll_top_family_*` con la familia más coleccionada por jugador.",
    "- **Investigación H2**: aunque H2 confirmada, los pocos casos donde `updated_at != created_at` pueden indicar trades/upgrades — mirar en Fase 2 si conviene una feature `coll_n_updated_items`.",
    "",
    "---", "",
    "## Diagrama del pipeline",
    "",
    "```",
    f"user_items_collection.csv ({n_before:,} filas × 5 cols, 844 MB)",
    "    │",
    f"    ├─ Lectura por chunks (size={CHUNKSIZE:,})",
    f"    ├─ Drop columna `_id`",
    f"    ├─ Filtrar por sample_user_ids dentro de cada chunk   (-{n_before - n_in_sample:,})",
    "    └─ Parsear created_at / updated_at a UTC",
    "",
    f"Filtrado ({n_in_sample:,} filas)",
    "    │",
    "    ├─ Validar H1, H2, H3, H4",
    "    ├─ Agregaciones por user_id (groupby una sola vez)",
    "    │   ├─ count, min, max, sum (30d/90d)",
    "    │   └─ nunique(family = id // 100)",
    "    └─ Reindex con sample_user_ids + fillna",
    "",
    f"features_user_items_collection_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]})",
    "```",
    "",
    "---", "",
    "## Reproducibilidad",
    "",
    "1. Ejecutar 02a_users.ipynb primero (genera sample_user_ids)",
    "2. Ejecutar 02l_user_items_collection.ipynb",
    "3. Verificar: features_user_items_collection_cutoff.parquet == 126.253 filas × 9 cols",
    "",
    "---", "",
    "## Referencias",
    "",
    "- PLANTILLA_NOTEBOOK_02.md — estructura estándar de notebook Fase 1.",
    "- 02g_devices.ipynb — referencia para SENTINEL_DAYS y point-in-time temporal.",
    "- 02j_fights_log.ipynb — referencia para tabla pesada con USECOLS y char_to_user (no aplicable aquí: user_id directo).",
    "",
]

report_path = REPORT_DIR / 'execution_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f"Informe guardado: {report_path}")
log_step('REPORT', 'execution_report.md generado')


Informe guardado: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/user_items_collection/execution_report.md
[REPORT] 13:13:55 — execution_report.md generado


In [19]:
# [REPORT] Resumen final en consola
elapsed = time.time() - NOTEBOOK_START

print("=" * 70)
print("RESUMEN FINAL — Notebook 02l_user_items_collection")
print("=" * 70)
print(f"  Tiempo total                : {int(elapsed//60)}m {int(elapsed%60)}s")
print(f"  Filas CSV original          : {n_before:,}")
print(f"  Filas en sample             : {n_in_sample:,} ({n_in_sample/n_before*100:.2f}%)")
print(f"  Cobertura del sample        : {coverage_pct:.2f}%")
print(f"  Filas features final        : {len(features):,} (== {N_SAMPLE:,} sample)")
print(f"  Columnas features           : {features.shape[1]}")
print()
print(f"  Usuarios con ítems          : {n_users_with_items_final:,} ({n_users_with_items_final/N_SAMPLE*100:.2f}%)")
print(f"  Usuarios sin ítems          : {n_users_no_items_final:,} ({n_users_no_items_final/N_SAMPLE*100:.2f}%)")
print()
print(f"  coll_total_items            : mean={features['coll_total_items'].mean():.1f}, max={features['coll_total_items'].max():,}, p99={features['coll_total_items'].quantile(0.99):.0f}")
print(f"  coll_unique_families        : mean={features['coll_unique_families'].mean():.2f}, max={features['coll_unique_families'].max():,}")
print(f"  coll_items_last_30d (mean)  : {features['coll_items_last_30d'].mean():.2f}")
print(f"  coll_items_last_90d (mean)  : {features['coll_items_last_90d'].mean():.2f}")
print()
print(f"  Hipótesis: H1={H1_STATE} ({H1_VALUE:.2f}%), H2={H2_STATE} ({H2_VALUE:.2f}%), H3={H3_STATE} (med={H3_MEDIAN}/max={H3_MAX}), H4={H4_STATE} (med batch={H4_MEDIAN})")
print()
print("Archivos generados:")
print(f"  features_user_items_collection_cutoff.parquet ({PARQUET_FEATURES.stat().st_size/1e6:.2f} MB)")
print(f"  execution_report.md")
print(f"  Gráficos y CSVs en {REPORT_DIR}")
print("=" * 70)


RESUMEN FINAL — Notebook 02l_user_items_collection
  Tiempo total                : 0m 12s
  Filas CSV original          : 7,796,748
  Filas en sample             : 876,284 (11.24%)
  Cobertura del sample        : 80.67%
  Filas features final        : 25,200 (== 25,200 sample)
  Columnas features           : 9

  Usuarios con ítems          : 20,329 (80.67%)
  Usuarios sin ítems          : 4,871 (19.33%)

  coll_total_items            : mean=26.1, max=223, p99=136
  coll_unique_families        : mean=1.66, max=4
  coll_items_last_30d (mean)  : 9.90
  coll_items_last_90d (mean)  : 16.36

  Hipótesis: H1=refutada (80.67%), H2=validada (100.00%), H3=validada (med=24.0/max=223), H4=refutada (med batch=1.0)

Archivos generados:
  features_user_items_collection_cutoff.parquet (1.16 MB)
  execution_report.md
  Gráficos y CSVs en /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/user_items_collection


In [20]:
# [REPORT] Logging dual: HTML + sección enriquecida del report
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from notebook_logging_template import (
    export_notebook_to_html, build_enriched_report_section,
)

notebook_path = PROJECT_ROOT / 'notebooks' / 'fase1_cleaning' / '02l_user_items_collection.ipynb'
html_path = REPORT_DIR / '02l_user_items_collection_run.html'
export_notebook_to_html(notebook_path, html_path)

enriched = build_enriched_report_section(
    df_final=features,
    raw_shape=(n_before, 5),
    filter_steps=[
        ('CSV original', n_before),
        ('En sample', n_in_sample),
    ],
)
with open(REPORT_DIR / 'execution_report.md', 'a', encoding='utf-8') as f:
    f.write('\n\n---\n\n' + enriched)
print(f"Enriquecimiento appendeado a {REPORT_DIR / 'execution_report.md'}")


HTML generado: 02l_user_items_collection_run.html (0.5 MB)
Enriquecimiento appendeado a /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/user_items_collection/execution_report.md
